In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import interact

# --- Simulation Function ---
def simulate_solar_system(
    c=5,
    battery_capacity_Wh=30,
    num_batteries=1,
    sleep_power_W=0.1,
    measuring_power_W=1,
    measuring_time_min=1,
    sending_power_W=10,
    sending_time_min=1,
    num_solar_cells=1,
    solar_spring_Wh=1,
    solar_summer_Wh=1.5,
    solar_autumn_Wh=1,
    solar_winter_Wh=0.8,
    simulation_days=30
):
    # Constants
    hours_per_day = 24
    minutes_per_day = hours_per_day * 60
    time_step_min = 1  # 1-minute resolution
    total_minutes = simulation_days * minutes_per_day
    
    # Create time array
    time = np.arange(total_minutes)  # in minutes
    hour_of_day = (time % minutes_per_day) / 60  # hours in day

    # Determine season per day
    days = np.arange(simulation_days)
    season_length = simulation_days // 4
    season_indices = np.zeros(simulation_days, dtype=int)
    season_indices[season_length:2*season_length] = 1
    season_indices[2*season_length:3*season_length] = 2
    season_indices[3*season_length:] = 3
    solar_season = np.repeat(season_indices, minutes_per_day)

    # Solar power per hour (Wh/hour)
    solar_hourly_Wh = np.zeros(total_minutes)
    solar_daily_Wh = [solar_spring_Wh, solar_summer_Wh, solar_autumn_Wh, solar_winter_Wh]
    
    for i in range(4):
        # distribute daily solar energy from 7h to 21h
        day_mask = (solar_season == i)
        for minute in range(minutes_per_day):
            hour = minute // 60
            if 7 <= hour < 21:
                solar_hourly_Wh[day_mask][minute::minutes_per_day] = solar_daily_Wh[i] / 14 / 60  # Wh per minute

    solar_hourly_W = solar_hourly_Wh * num_solar_cells  # W per minute

    # Initialize consumption arrays
    sleep_W = np.zeros(total_minutes)
    measuring_W = np.zeros(total_minutes)
    sending_W = np.zeros(total_minutes)

    # Schedule cycles
    cycle_length_min = measuring_time_min + sending_time_min
    for day in range(simulation_days):
        for cycle in range(c):
            start_minute = day * minutes_per_day + cycle * cycle_length_min
            measuring_start = start_minute
            measuring_end = start_minute + measuring_time_min
            sending_start = measuring_end
            sending_end = sending_start + sending_time_min

            measuring_W[measuring_start:measuring_end] = measuring_power_W
            sending_W[sending_start:sending_end] = sending_power_W

    # Sleep power when nothing else is active
    sleep_W[(measuring_W == 0) & (sending_W == 0)] = sleep_power_W

    # Total consumption
    total_consumption_W = measuring_W + sending_W + sleep_W

    # Energy integration
    battery_capacity_total_Wh = battery_capacity_Wh * num_batteries
    battery_energy_Wh = np.zeros(total_minutes)
    energy = 0

    for t in range(total_minutes):
        net_power_W = solar_hourly_W[t] - total_consumption_W[t]  # W
        energy += net_power_W / 60  # convert W*min to Wh
        # Limit to battery capacity
        if energy > battery_capacity_total_Wh:
            energy = battery_capacity_total_Wh
        if energy < 0:
            energy = 0
        battery_energy_Wh[t] = energy

    # --- Plotting ---
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Battery Energy Storage (Wh)", "Power Flows (W)"))

    # Battery subplot
    fig.add_trace(go.Scatter(x=time/60, y=battery_energy_Wh, name="Battery Energy"), row=1, col=1)

    # Power subplot
    fig.add_trace(go.Scatter(x=time/60, y=solar_hourly_W, name="Solar Generation"), row=2, col=1)
    fig.add_trace(go.Scatter(x=time/60, y=measuring_W, name="Measuring Consumption"), row=2, col=1)
    fig.add_trace(go.Scatter(x=time/60, y=sending_W, name="Sending Consumption"), row=2, col=1)
    fig.add_trace(go.Scatter(x=time/60, y=sleep_W, name="Sleep Consumption"), row=2, col=1)

    fig.update_layout(height=700, width=900, title_text="Solar-Powered System Simulation")
    fig.update_xaxes(title_text="Time (hours)", row=2, col=1)
    fig.update_yaxes(title_text="Wh", row=1, col=1)
    fig.update_yaxes(title_text="W", row=2, col=1)
    
    fig.show()

# --- Sliders / Interactive Widgets ---
interact(simulate_solar_system,
         c=widgets.IntSlider(min=1, max=50, step=1, value=5, description="Cycles/day"),
         battery_capacity_Wh=widgets.FloatSlider(min=1, max=100, step=1, value=30, description="Battery Cap. (Wh)"),
         num_batteries=widgets.IntSlider(min=1, max=5, step=1, value=1, description="Num Batteries"),
         sleep_power_W=widgets.FloatSlider(min=0.01, max=1, step=0.01, value=0.1, description="Sleep Power (W)"),
         measuring_power_W=widgets.FloatSlider(min=0.1, max=10, step=0.1, value=1, description="Measuring Power (W)"),
         measuring_time_min=widgets.IntSlider(min=1, max=30, step=1, value=1, description="Measuring Time (min)"),
         sending_power_W=widgets.FloatSlider(min=1, max=20, step=1, value=10, description="Sending Power (W)"),
         sending_time_min=widgets.IntSlider(min=1, max=30, step=1, value=1, description="Sending Time (min)"),
         num_solar_cells=widgets.IntSlider(min=1, max=10, step=1, value=1, description="Num Solar Cells"),
         solar_spring_Wh=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1, description="Spring Solar Wh"),
         solar_summer_Wh=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1.5, description="Summer Solar Wh"),
         solar_autumn_Wh=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1, description="Autumn Solar Wh"),
         solar_winter_Wh=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=0.8, description="Winter Solar Wh"),
         simulation_days=widgets.IntSlider(min=1, max=365, step=1, value=30, description="Days"))


interactive(children=(IntSlider(value=5, description='Cycles/day', max=50, min=1), FloatSlider(value=30.0, des…

<function __main__.simulate_solar_system(c=5, battery_capacity_Wh=30, num_batteries=1, sleep_power_W=0.1, measuring_power_W=1, measuring_time_min=1, sending_power_W=10, sending_time_min=1, num_solar_cells=1, solar_spring_Wh=1, solar_summer_Wh=1.5, solar_autumn_Wh=1, solar_winter_Wh=0.8, simulation_days=30)>